# Base Model vs Finetuned Model Comparison

This notebook runs the complete comparison experiment:
1. Train probes on **base model** (distilbert-base-uncased)
2. Run interventions on **base model** using trained probes
3. Run interventions on **finetuned model** (distilbert-base-uncased-finetuned-sst-2-english)
4. Compare results

**Goal**: Understand how finetuning changes where negation information is encoded in the model.

**Before running**: 
- Make sure you've run `00_setup_colab.ipynb` and `01_download_data.ipynb`
- Enable GPU: Runtime → Change runtime type → GPU


In [ ]:
# Configuration
import os
import sys
from pathlib import Path
from datetime import datetime

# Add project to path
if 'Negation-Origin-Tracing' in os.getcwd():
    sys.path.insert(0, os.getcwd())
elif os.path.exists('Negation-Origin-Tracing'):
    os.chdir('Negation-Origin-Tracing')
    sys.path.insert(0, os.getcwd())

# Configuration
BASE_MODEL = "distilbert-base-uncased"
FINETUNED_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
DATA_DIR = "data/raw"
OUTPUT_DIR = f"experiments/comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
BATCH_SIZE = 16
MAX_EPOCHS = 10
PROBE_LR = 1e-3

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/base_probes", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/base_interventions", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/finetuned_interventions", exist_ok=True)

print(f"✓ Configuration set")
print(f"  Base model: {BASE_MODEL}")
print(f"  Finetuned model: {FINETUNED_MODEL}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {MAX_EPOCHS}")


## Step 1: Train Probes on Base Model

Train probes on all layers of the base (non-finetuned) model to identify where negation information is encoded.


In [ ]:
# Step 1: Train probes on base model
import subprocess
import sys

print("=" * 60)
print("STEP 1: Training probes on base model")
print("=" * 60)
print()

# Run layer search script
cmd = [
    sys.executable,
    "src/scripts/search_layers.py",
    "--model_name", BASE_MODEL,
    "--mode", "probe",
    "--data_dir", DATA_DIR,
    "--batch_size", str(BATCH_SIZE),
    "--max_epochs", str(MAX_EPOCHS),
    "--probe_lr", str(PROBE_LR),
    "--layers", "all",
    "--pooling_strategies", "all",
    "--output_dir", f"{OUTPUT_DIR}/base_probes",
    "--seed", "42",
    "--devices", "1",  # Use GPU if available
]

result = subprocess.run(cmd, check=True)
print("\n✓ Probe training complete!")


## Step 2: Identify Best Layer

Find the layer with the best probe performance (highest test AUROC).


In [ ]:
# Step 2: Identify best layer from probe results
import json
import pandas as pd

print("=" * 60)
print("STEP 2: Identifying best layer")
print("=" * 60)
print()

# Load probe results
probe_results_path = f"{OUTPUT_DIR}/base_probes/results_summary.json"
if not os.path.exists(probe_results_path):
    raise FileNotFoundError(f"Probe results not found at {probe_results_path}")

with open(probe_results_path, 'r') as f:
    probe_results = json.load(f)

# Find best layer by test_auroc
df = pd.DataFrame(probe_results)
best_result = df.loc[df['test_auroc'].idxmax()]

BEST_LAYER = int(best_result['layer_idx'])
BEST_POOLING = best_result['pooling_strategy']
BEST_CKPT = best_result.get('checkpoint_path', '')

print(f"Best layer: {BEST_LAYER}")
print(f"Best pooling strategy: {BEST_POOLING}")
print(f"Best test AUROC: {best_result['test_auroc']:.4f}")
print(f"Best test accuracy: {best_result['test_accuracy']:.4f}")
print(f"Checkpoint: {BEST_CKPT}")

# Display top 3 layers
print("\nTop 3 layers by test AUROC:")
top_3 = df.nlargest(3, 'test_auroc')[['layer_idx', 'pooling_strategy', 'test_auroc', 'test_accuracy']]
print(top_3.to_string(index=False))


## Step 3: Run Interventions on Base Model

Run activation patching experiments on the base model using the trained probes.


In [ ]:
# Step 3: Run interventions on base model (with probes)
import torch

print("=" * 60)
print("STEP 3: Running interventions on BASE model")
print("=" * 60)
print()

skip_base = False
negation_data_path = f"{DATA_DIR}/test/negation.parquet"

if not os.path.exists(negation_data_path):
    print(f"⚠ Warning: Negation dataset not found at {negation_data_path}")
    print("  Skipping base model interventions")
    print("  Make sure you've run 01_download_data.ipynb")
    skip_base = True
else:
    if not os.path.exists(BEST_CKPT):
        print(f"⚠ Warning: Checkpoint not found at {BEST_CKPT}")
        # Try to find checkpoint
        import glob
        checkpoints = glob.glob(f"{OUTPUT_DIR}/base_probes/**/*.ckpt", recursive=True)
        if checkpoints:
            BEST_CKPT = checkpoints[0]
            print(f"  Using checkpoint: {BEST_CKPT}")
        else:
            print("  No checkpoint found. Skipping base model interventions.")
            skip_base = True
    
    if not skip_base:
        cmd = [
            sys.executable,
            "src/scripts/run_interventions.py",
            "--model_ckpt", BEST_CKPT,
            "--model_name", BASE_MODEL,
            "--mode", "probe",
            "--probe_layer", str(BEST_LAYER),
            "--data_path", negation_data_path,
            "--intervention_type", "activation_patching",
            "--layers", str(BEST_LAYER),
            "--batch_size", str(BATCH_SIZE),
            "--output_dir", f"{OUTPUT_DIR}/base_interventions",
            "--device", "cuda" if torch.cuda.is_available() else "cpu",
        ]
        
        result = subprocess.run(cmd, check=True)
        print("\n✓ Base model interventions complete!")


## Step 4: Run Interventions on Finetuned Model

Run the same activation patching experiments on the finetuned model for comparison.


In [ ]:
# Step 4: Run interventions on finetuned model
import torch

print("=" * 60)
print("STEP 4: Running interventions on FINETUNED model")
print("=" * 60)
print()

negation_data_path = f"{DATA_DIR}/test/negation.parquet"
if not os.path.exists(negation_data_path):
    print(f"⚠ Warning: Negation dataset not found at {negation_data_path}")
    print("  Skipping finetuned model interventions")
else:
    # Run on the same layer(s) as base model for comparison
    cmd = [
        sys.executable,
        "src/scripts/run_interventions.py",
        "--mode", "finetune",
        "--model_name", FINETUNED_MODEL,
        "--data_path", negation_data_path,
        "--intervention_type", "activation_patching",
        "--layers", str(BEST_LAYER),
        "--batch_size", str(BATCH_SIZE),
        "--output_dir", f"{OUTPUT_DIR}/finetuned_interventions",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    
    result = subprocess.run(cmd, check=True)
    print("\n✓ Finetuned model interventions complete!")


## Step 5: Generate Comparison Summary

Combine all results into a comparison summary.


In [ ]:
# Step 5: Generate comparison summary
print("=" * 60)
print("STEP 5: Generating comparison summary")
print("=" * 60)
print()

summary = {
    'experiment_config': {
        'base_model': BASE_MODEL,
        'finetuned_model': FINETUNED_MODEL,
        'best_layer': BEST_LAYER,
        'best_pooling': BEST_POOLING,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
    },
    'probe_results': None,
    'base_interventions': None,
    'finetuned_interventions': None,
}

# Load probe results
if os.path.exists(probe_results_path):
    with open(probe_results_path, 'r') as f:
        probe_results = json.load(f)
        summary['probe_results'] = {
            'total_experiments': len(probe_results),
            'best_layer': BEST_LAYER,
            'best_auroc': float(best_result['test_auroc']),
            'best_accuracy': float(best_result['test_accuracy']),
        }

# Load base interventions
base_int_file = f"{OUTPUT_DIR}/base_interventions/intervention_results.json"
if os.path.exists(base_int_file):
    with open(base_int_file, 'r') as f:
        summary['base_interventions'] = json.load(f)

# Load finetuned interventions
finetuned_int_file = f"{OUTPUT_DIR}/finetuned_interventions/intervention_results.json"
if os.path.exists(finetuned_int_file):
    with open(finetuned_int_file, 'r') as f:
        summary['finetuned_interventions'] = json.load(f)

# Save summary
summary_file = f"{OUTPUT_DIR}/comparison_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✓ Comparison summary saved to: {summary_file}")

# Display summary
print("\n" + "=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"\nBest Layer: {BEST_LAYER} ({BEST_POOLING} pooling)")
print(f"Best Probe AUROC: {best_result['test_auroc']:.4f}")
print(f"Best Probe Accuracy: {best_result['test_accuracy']:.4f}")

if summary['base_interventions']:
    print("\nBase Model Interventions:")
    for exp_type, results in summary['base_interventions'].items():
        for layer, stats in results.items():
            print(f"  {exp_type} - Layer {layer}:")
            print(f"    Avg label flips: {stats.get('avg_label_flips', 'N/A')}")
            print(f"    Avg logit delta: {stats.get('avg_logit_delta', 'N/A')}")

if summary['finetuned_interventions']:
    print("\nFinetuned Model Interventions:")
    for exp_type, results in summary['finetuned_interventions'].items():
        for layer, stats in results.items():
            print(f"  {exp_type} - Layer {layer}:")
            print(f"    Avg label flips: {stats.get('avg_label_flips', 'N/A')}")
            print(f"    Avg logit delta: {stats.get('avg_logit_delta', 'N/A')}")

print(f"\n✓ All results saved to: {OUTPUT_DIR}")


In [ ]:
# Optional: Create zip file for download
import shutil

try:
    zip_path = f"{OUTPUT_DIR}.zip"
    shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
    print(f"✓ Results zipped: {zip_path}")
    print(f"  You can download it from Colab's file browser")
except Exception as e:
    print(f"⚠ Could not create zip: {e}")
    print(f"  You can download files individually from: {OUTPUT_DIR}")

# List all output files
print(f"\nOutput files in {OUTPUT_DIR}:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:  # Show first 10 files
        print(f"{subindent}{file}")
    if len(files) > 10:
        print(f"{subindent}... and {len(files) - 10} more files")
